<a href="https://colab.research.google.com/github/mikexcohen/Substack/blob/main/kink6174.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

|<h2>Substack post:</h2>|<h1><a href="https://mikexcohen.substack.com/p/the-fourier-transform-explained-with" target="_blank">A kink in the universe at 6174</a></h1>|
|-|:-:|
|<h2>Teacher:<h2>|<h1>Mike X Cohen, <a href="https://sincxpress.com" target="_blank">sincxpress.com</a></h1>|

<br>

<i>Using the code without reading the post may lead to confusion or errors.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
### Run this cell only if you're using "dark mode"

# svg plots (higher-res)
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

plt.rcParams.update({
    'figure.facecolor': '#171717',
    'figure.edgecolor': '#171717',
    'axes.facecolor':   '#171717',
    'axes.edgecolor':   '#DDE2F4',
    'axes.labelcolor':  '#DDE2F4',
    'xtick.color':      '#DDE2F4',
    'ytick.color':      '#DDE2F4',
    'text.color':       '#DDE2F4',
    'axes.spines.right': False,
    'axes.spines.top':   False,
    'axes.titleweight': 'bold',
    'axes.labelweight': 'bold',
})

# **A function for one iteration**

In [ ]:
def subtract_ordered(x):

  # pad string so that 42 -> '0042'
  s = str(x).zfill(4)

  # reorderings
  large2small = int(''.join(sorted(s,reverse=True)))
  small2large = int(''.join(sorted(s)))

  # subtract
  return large2small - small2large

# **Example with one number**

In [ ]:
# target number
kaprekar = 6174

# starting number
number = 4152

while number != kaprekar:
  number = subtract_ordered(number)
  print(number)

# **Looping over all 4-digit integers**

In [ ]:
numbers = np.arange(10000)

data = []

for N in numbers:

  # skip this number if all digits are the same
  if len(set(str(N).zfill(4))) == 1:
    continue

  # initialize
  i,num = 0,N

  # algorithm
  while num != kaprekar:

    # apply the algo and increment the counter
    num = subtract_ordered(num)
    i += 1

  # store the number, iterations, and  counts,
  data.append([N,i,len(set(str(N)))])

# convenient to have numpy
data = np.stack(data,dtype=int)
n = data.shape[0]

# **Visualizations**

In [ ]:
plt.figure(figsize=(12,4))
plt.scatter(data[:,0],
            data[:,1]+np.random.normal(0,.07,n),
            50,c=data[:,1],alpha=.2,cmap='jet',edgecolor=[.7,.7,.7],
            )
plt.gca().set(xlabel='Number', ylabel='Number of iterations',
              xlim=[-30,10030], title=f'How many iterations to reach {kaprekar}')
plt.show()

In [ ]:
y = np.bincount(data[:,1])

plt.figure(figsize=(10,4))
plt.bar(range(8),y,color=plt.cm.plasma(y/y.max()),edgecolor='w')
plt.gca().set(xlabel='Number of iterations',ylabel='"Depth" count')
plt.show()

In [ ]:
# transition matrix
T = np.zeros((8,8))
for i in range(n-1):
  T[data[i+1,1],data[i,1]] += 1
T /= T.sum() # normalize
T[T==0] = np.nan # for plotting

# transition coordinates
x = data[:-1,1] + np.random.uniform(-.2,.2,n-1)
y = data[1:,1]  + np.random.uniform(-.2,.2,n-1)

# show them
fig,axs = plt.subplots(1,2,figsize=(12,5))
axs[0].scatter(x,y,30,c=np.random.rand(n-1),alpha=.25,cmap='Purples')
axs[0].plot(data[:-1,1],data[1:,1],'w',alpha=.1,zorder=-100)
axs[0].set(xlabel='Iterations at N',ylabel='Iterations at N+1')

h = axs[1].imshow(T,cmap='plasma',origin='lower',vmin=.0,vmax=.06)
fig.colorbar(h,ax=axs[1],pad=.01)
axs[1].set(xlabel='Iterations at N',ylabel='Iterations at N+1',title='Transition matrix')

plt.tight_layout()
plt.savefig('4cover.svg')
plt.show()

In [ ]:
T - T.T # a symmetric matrix equals its transpose

In [ ]:
N = 1234
subtract_ordered(N),subtract_ordered(9999-N)

In [ ]:
# recalculate T without nan's
T = np.zeros((8,8))
for i in range(n-1):
  T[data[i+1,1],data[i,1]] += 1
T /= T.sum() # normalize

# eigendecomposition
d,w = np.linalg.eig(T)
sidx = np.argsort(d)[::-1]
eigvals = d[sidx]
eigvecs = w[sidx]

plt.figure(figsize=(9,3))
plt.plot(eigvals,'wh',markersize=14,markerfacecolor=[.7,.9,.7])
plt.axhline(0,color=[.3,.3,.3],linestyle='--',linewidth=.5,zorder=-1)
plt.gca().set(xlabel='Component',ylabel='Eigenvalue')
plt.show()

In [ ]:
plt.figure(figsize=(8,3))

# plot the violins
for i in range(1,8):
  v = plt.violinplot(np.sqrt(np.diff(np.where(data[:,1]==i)[0])),positions=[i])

  v['bodies'][0].set_facecolor(plt.cm.plasma(i/7))
  v['bodies'][0].set_alpha(1),
  v['cbars'].set_edgecolor([.7,.7,.9,.4])
  v['cmins'].set_edgecolor([.9,.7,.7,.4])
  v['cmaxes'].set_edgecolor([.7,.9,.7,.4])

plt.gca().set(ylabel='Digits until same depth (sqrt)', xlabel='Number of iterations',
              xlim=[0,8], xticks=range(1,8), yticks=range(0,21,5),yticklabels=np.arange(0,21,5)**2)
plt.show()